# L57 · MLP 从零搭建 — 手写全连接层、激活函数、前向 / 反向完整实现

**目标**：实现 `Neuron → Layer → MLP`，每层支持前向传播和反向传播（`backward()`）。

🔗 **Aurora 连接**：这个 MLP 和 PyTorch `nn.Module` 完全同构；`p3_nn.ipynb` 将用 PyTorch 复现相同的三层结构，届时你可以逐行对照两份代码，验证自己的实现和框架行为一致。

## 核心直觉

神经网络就是把「可微的标量运算」组合成树状计算图。`Value` 已经记录了每个节点的梯度传播规则；现在只差一件事：**按照「输入维度→隐藏层→输出维度」的顺序，把一批 `Value` 对象连接起来**。`Neuron` 是最小单元（一个加权求和），`Layer` 是一行 Neuron，`MLP` 是多行 Layer 串成的管道。构建好之后，调一次 `loss.backward()` 就能把梯度反推到所有权重。

In [ ]:
import random

# ── Value 类（来自 a3_autograd.ipynb，这里直接粘贴以便本节独立运行）──
import math

class Value:
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = float(data)
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __radd__(self, other): return self + other

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def __rmul__(self, other): return self * other

    def __neg__(self): return self * -1
    def __sub__(self, other): return self + (-other)
    def __rsub__(self, other): return Value(other) + (-self)
    def __truediv__(self, other): return self * Value(other.data**-1 if isinstance(other, Value) else other**-1)

    def __pow__(self, exp):
        assert isinstance(exp, (int, float))
        out = Value(self.data**exp, (self,), f'**{exp}')
        def _backward():
            self.grad += exp * (self.data**(exp-1)) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), 'tanh')
        def _backward():
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward
        return out

    def relu(self):
        out = Value(max(0, self.data), (self,), 'relu')
        def _backward():
            self.grad += (out.data > 0) * out.grad
        out._backward = _backward
        return out

    def backward(self):
        topo, visited = [], set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build(child)
                topo.append(v)
        build(self)
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()

# 快速冒烟测试
a = Value(2.0); b = Value(3.0)
c = (a * b + Value(1.0)).tanh()
c.backward()
assert abs(a.grad) > 0, "Value.backward() 未工作"
print("✅ Value 类加载成功")

## 1. Neuron — 单个神经元

一个 `Neuron(nin)` 持有 `nin` 个权重 `w` 和 1 个偏置 `b`，都是 `Value` 对象（初始化为 `[-1, 1)` 的随机数）。前向传播的数学表达式是：

```
output = activation( w[0]*x[0] + w[1]*x[1] + ... + w[nin-1]*x[nin-1] + b )
```

`activation` 默认用 `tanh`（`nonlin=True`），关闭时输出线性值（线性输出层用）。因为 `w` 和 `b` 都是 `Value`，乘法和加法自动记录梯度链路——调用 `output.backward()` 即可把梯度推回到所有 `w` 和 `b`。

In [ ]:
# 演示：手动搭一个 nin=2 的 Neuron（还没有 class，先把运算写出来）
random.seed(42)
w = [Value(random.uniform(-1, 1)) for _ in range(2)]
b  = Value(0.0)

x  = [Value(1.5), Value(-0.5)]

# 前向：dot + bias + tanh
act = sum((wi * xi for wi, xi in zip(w, x)), Value(0.0)) + b
out = act.tanh()

print("w  =", [round(wi.data, 4) for wi in w])
print("b  =", round(b.data, 4))
print("x  =", [xi.data for xi in x])
print("act=", round(act.data, 4))
print("out=", round(out.data, 4), "  (tanh 值在 (-1, 1) 之间)")

## 2. Layer — 并行的一组 Neuron

`Layer(nin, nout)` 包含 `nout` 个独立的 `Neuron(nin)`，它们接收**同一个输入向量 `x`**，各自独立计算，输出 `nout` 个标量组成的列表：

```
[neuron_0(x), neuron_1(x), ..., neuron_{nout-1}(x)]
```

`nout=1` 时输出列表长度为 1（最终回归层的常见设置）。每个 `Neuron` 的参数相互独立，梯度互不干扰。

In [ ]:
# 演示：手动搭 nout=3 的 Layer（nin=2 → 输出 3 个标量）
random.seed(0)
nin, nout = 2, 3
# 3 个 Neuron，每个有 2 个权重
neurons_demo = [
    [Value(random.uniform(-1, 1)) for _ in range(nin)]
    for _ in range(nout)
]
biases_demo = [Value(0.0) for _ in range(nout)]

x_demo = [Value(1.0), Value(-1.0)]

outputs_demo = []
for ws, b in zip(neurons_demo, biases_demo):
    act = sum((w * xi for w, xi in zip(ws, x_demo)), Value(0.0)) + b
    outputs_demo.append(act.tanh())

print(f"Layer(nin=2, nout=3) 输出 {len(outputs_demo)} 个 Value:")
for i, o in enumerate(outputs_demo):
    print(f"  out[{i}] = {o.data:.4f}")

## 3. MLP — 多层串联

`MLP(nin, nouts)` 把 `len(nouts)` 个 `Layer` 首尾相连：第 0 层接收原始输入（维度 `nin`），第 `k` 层接收第 `k-1` 层的输出。每层的 `nin` 恰好是上一层的 `nout`：

```
Layer(nin,       nouts[0])   → 输出 nouts[0] 维
Layer(nouts[0],  nouts[1])   → 输出 nouts[1] 维
...
Layer(nouts[-2], nouts[-1])  → 最终输出
```

最后一层通常关闭非线性（`nonlin=False`）。`parameters()` 遍历所有层的所有 `Neuron`，把全部 `w` 和 `b` 拍成一个平坦列表——训练时对这个列表里每个 `Value` 执行 `p.data -= lr * p.grad` 就完成一步梯度下降。

In [ ]:
# 演示：用嵌套列表模拟 MLP(3, [4, 4, 1]) 的层结构
# （先打印结构，后面才实现真正的 class）
layer_shapes = [(3, 4), (4, 4), (4, 1)]
total_params = sum(nin * nout + nout for nin, nout in layer_shapes)
print("MLP(3, [4, 4, 1]) 参数统计：")
for i, (n_in, n_out) in enumerate(layer_shapes):
    w_count = n_in * n_out
    b_count = n_out
    print(f"  Layer {i}: {n_in}×{n_out} 权重 + {n_out} 偏置 = {w_count + b_count}")
print(f"  总计：{total_params} 个可训练参数")

## 4. ✏️ 实现 `class Neuron, class Layer, class MLP`

签名：`MLP(nin, nouts)` where `nouts=[h1, h2, ..., n_out]`

**推理路线**：
1. `Neuron(nin, nonlin=True)`：`self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]`，`self.b = Value(0.0)`；`__call__(x)` 先算 `act = sum(wi*xi) + b`，若 `nonlin=True` 返回 `act.tanh()`，否则直接返回 `act`。
2. `Layer(nin, nout, **kwargs)`：`self.neurons = [Neuron(nin, **kwargs) for _ in range(nout)]`；`__call__(x)` 返回 `[n(x) for n in self.neurons]`，若只有 1 个 Neuron 则解包成标量（`outs[0]`）。
3. `MLP(nin, nouts)`：用 `zip([nin]+nouts[:-1], nouts)` 得到每层的 `(nin, nout)` 对，最后一层加 `nonlin=False`；`__call__(x)` 依次把输出喂给下一层；`parameters()` 对每层每个 Neuron 调 `n.parameters()` 并 `extend` 到一个列表里返回。

**参考输入输出**：
```python
random.seed(0)
m = MLP(3, [4, 4, 1])
x = [Value(1.0), Value(2.0), Value(3.0)]
out = m(x)
# out 是单个 Value（nouts[-1]==1 时解包），out.data 是一个标量
# len(m.parameters()) == 41
```

<details><summary>点击查看参考实现</summary>

```python
class Neuron:
    def __init__(self, nin, nonlin=True):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(0.0)
        self.nonlin = nonlin

    def __call__(self, x):
        act = sum((wi * xi for wi, xi in zip(self.w, x)), Value(0.0)) + self.b
        return act.tanh() if self.nonlin else act

    def parameters(self):
        return self.w + [self.b]

    def __repr__(self):
        kind = "Tanh" if self.nonlin else "Linear"
        return f"{kind}Neuron({len(self.w)})"


class Layer:
    def __init__(self, nin, nout, **kwargs):
        self.neurons = [Neuron(nin, **kwargs) for _ in range(nout)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

    def __repr__(self):
        return f"Layer([{', '.join(str(n) for n in self.neurons)}])"


class MLP:
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [
            Layer(sz[i], sz[i+1], nonlin=(i < len(nouts)-1))
            for i in range(len(nouts))
        ]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x if isinstance(x, list) else [x]

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

    def __repr__(self):
        return f"MLP([{', '.join(str(l) for l in self.layers)}])"
```

</details>

In [ ]:
class Neuron:
    def __init__(self, nin, nonlin=True):
        # ✏️ TODO: 初始化 self.w（nin 个 Value，均匀随机 [-1, 1)）和 self.b（Value(0.0)）
        # ✏️ TODO: 保存 self.nonlin
        pass

    def __call__(self, x):
        # ✏️ TODO: 计算 act = sum(wi * xi for ...) + self.b
        # ✏️ TODO: 若 self.nonlin 为 True，返回 act.tanh()，否则返回 act
        pass

    def parameters(self):
        # ✏️ TODO: 返回 self.w + [self.b]
        pass

    def __repr__(self):
        kind = "Tanh" if self.nonlin else "Linear"
        return f"{kind}Neuron({len(self.w)})"


class Layer:
    def __init__(self, nin, nout, **kwargs):
        # ✏️ TODO: self.neurons = nout 个 Neuron(nin, **kwargs)
        pass

    def __call__(self, x):
        # ✏️ TODO: outs = [n(x) for n in self.neurons]
        # ✏️ TODO: 若 len(outs)==1 返回 outs[0]，否则返回 outs
        pass

    def parameters(self):
        # ✏️ TODO: 返回所有 neuron 的 parameters() 拍平后的列表
        pass

    def __repr__(self):
        return f"Layer([{', '.join(str(n) for n in self.neurons)}])"


class MLP:
    def __init__(self, nin, nouts):
        # ✏️ TODO: sz = [nin] + nouts
        # ✏️ TODO: self.layers = Layer(sz[i], sz[i+1], nonlin=...) 列表
        #          最后一层 nonlin=False，其余 True
        pass

    def __call__(self, x):
        # ✏️ TODO: 依次把 x 喂入每一层（x = layer(x)）
        # ✏️ TODO: 统一返回列表（若结果不是 list 则 [x]）
        pass

    def parameters(self):
        # ✏️ TODO: 返回所有层所有参数的平坦列表
        pass

    def __repr__(self):
        return f"MLP([{', '.join(str(l) for l in self.layers)}])"


In [ ]:
random.seed(0)

# 基础结构检查
m = MLP(2, [3, 1])
out = m([Value(0.5), Value(-0.5)])
assert isinstance(out, list) and len(out) == 1, "MLP 输出应为长度 1 的列表"
assert isinstance(out[0], Value), "MLP 输出元素应为 Value"
print("✅ MLP(2,[3,1]) 前向传播通过，out[0].data =", round(out[0].data, 4))

# 参数数量检查
m2 = MLP(3, [4, 4, 1])
params = m2.parameters()
assert len(params) == 41, f"参数数量应为 41，实际 {len(params)}"
print("✅ MLP(3,[4,4,1]) 参数数量正确：", len(params))

# 反向传播检查
out2 = m2([Value(1.0), Value(2.0), Value(3.0)])
loss = out2[0]
loss.backward()
grads = [p.grad for p in params]
assert any(g != 0.0 for g in grads), "backward() 后参数梯度应不全为 0"
print("✅ backward() 梯度传播正常，首个参数梯度 =", round(params[0].grad, 6))

## 参数实验：验证 MLP(3,[4,4,1]) 的参数总数

手动推算三层的参数数量，再用 `len(m.parameters())` 核对：

| 层 | nin | nout | 权重数 `nin × nout` | 偏置数 `nout` | 小计 |
|----|-----|------|---------------------|---------------|------|
| 0  |  3  |  4   | 12                  | 4             | **16** |
| 1  |  4  |  4   | 16                  | 4             | **20** |
| 2  |  4  |  1   |  4                  | 1             | **5**  |
| 合计 | — | — | — | — | **41** |

预期现象：`len(m.parameters()) == 41`，且 `Neuron.__repr__` 最后一层显示 `LinearNeuron`（无非线性）。

In [ ]:
random.seed(99)
m3 = MLP(3, [4, 4, 1])
print("网络结构：", m3)
print()

for i, layer in enumerate(m3.layers):
    lp = layer.parameters()
    print(f"Layer {i}: {len(lp):3d} 个参数  |  neurons: {layer.neurons}")

total = len(m3.parameters())
print(f"\n总参数：{total}  （预期 41）")

# 检查最后一层是否关闭了非线性
assert not m3.layers[-1].neurons[0].nonlin, "最后一层 Neuron 应 nonlin=False（线性输出）"
print("✅ 最后一层为线性（nonlin=False）")

## 本课收束

本节实现了 `Neuron`（单个加权求和+激活）、`Layer`（并行 `nout` 个 Neuron）、`MLP`（多层串联），`parameters()` 返回全部 41 个可训练 `Value` 的平坦列表，供梯度下降逐一更新。这套结构和 PyTorch `nn.Module` 完全同构，对应 Aurora 训练管道 `aurora/train.py` 里的网络主体。下一节 `a5_train.ipynb` 将用这个 MLP 拟合一个玩具数据集，串起「前向 → 算 loss → backward() → 更新参数 → 清零梯度」的完整训练循环。